# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how Strands turns a function into a tool the model can see
- what a tool call looks like, and how Strands runs it for you
- how tool errors and safety checks work when the model gets something wrong

In [1]:
from strands import Agent

from agent_contracts.harness import verify_model
from strands_agents_impl.llm import make_model
from strands_agents_impl.prompts import SPECIALIST  # the shared instructions, in strands_agents_impl/prompts.py

# Check that Ollama is running and the shared model exists, then build the shared model (strands_agents_impl/llm.py).
verify_model()
model = make_model()  # the same model setup as notebook 01

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about Strands: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
# The shared tools live in contracts, not in Strands. TOOLS is the list of the five plain functions.
from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # start clean: clears the refund ledger, the tickets and the tool-call log
print("ShipFast's tools:", ", ".join(f.__name__ for f in shipfast.TOOLS))

ShipFast's tools: lookup_order, track_shipment, search_policy, issue_refund, escalate_to_human


Call one directly, like any function. `lookup_order` takes an order ID and returns a JSON string, which
the cell below parses so it prints as one field per line.

In [3]:
import json

# Call a tool by hand: the argument is an order ID, the return value is a JSON string.
order = json.loads(shipfast.lookup_order("A1001"))

for field, value in order.items():
    print(f"{field:13}{value}")

order_id     A1001
customer     Maria Santos
item         Wireless Headphones
total        89.0
status       in_transit
carrier      LBC
ordered_on   2026-10-28
eta          2026-11-04
delivered_on None


## Turning a function into a tool

Strands' `tool` reads a function's name, type hints and docstring and builds a **tool spec**: the name,
description and JSON input schema the model sees. `strands_agents_impl/tools.py` wraps all five:

```python
def shipfast_tools() -> list:
    # shipfast.TOOLS is the five shared functions: lookup_order, track_shipment, search_policy,
    # issue_refund and escalate_to_human. tool(f) reads each one's name, type hints and docstring into a
    # tool spec; calling the tool still runs the shared function, unchanged.
    return [tool(f) for f in shipfast.TOOLS]
```

In [ ]:
from strands_agents_impl.tools import shipfast_tools  # used in the cells below

Build the tools and look at what the model is told about each one: its name and its arguments, with their
types.

In [5]:
tools = shipfast_tools()  # one Strands tool per function

print(f"{'tool':18} arguments")
for t in tools:
    # the argument schema in the tool spec, built from the function's type hints
    properties = t.tool_spec["inputSchema"]["json"]["properties"]
    arguments = ", ".join(f"{name}: {spec['type']}" for name, spec in properties.items())
    print(f"{t.tool_name:18} {arguments}")

tool               arguments
lookup_order       order_id: string
track_shipment     order_id: string
search_policy      query: string
issue_refund       order_id: string, amount: number, reason: string
escalate_to_human  order_id: string, summary: string


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

Strands' `Agent` runs tools for you from the start, so to see a raw tool call we go one level down, to the
model itself. `model.stream(...)` sends one request with the tool specs and streams back the reply; a
tool call arrives as a `toolUse` block with the tool's name and its input. Nothing is run.

In [6]:
async def first_move(text: str) -> list[dict]:
    """The model's first reply to a message, as the tool calls it asks for (nothing is run)."""
    calls, current = [], None
    # stream() sends one request (the message, the tool specs, the instructions) and yields events as they arrive
    async for event in model.stream([{"role": "user", "content": [{"text": text}]}],
                                    tool_specs=[t.tool_spec for t in tools], system_prompt=SPECIALIST):
        start = event.get("contentBlockStart", {}).get("start", {})
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if "toolUse" in start:  # a tool call begins: its name arrives first
            current = {"name": start["toolUse"]["name"], "input": ""}
            calls.append(current)
        elif "toolUse" in delta and current is not None:  # its arguments arrive in pieces
            current["input"] += delta["toolUse"].get("input", "")
    return calls

Ask it about order A1001 and print the tool call it comes back with. If it answers in text
instead, which smaller models sometimes do, the cell says so.

In [7]:
calls = await first_move("Hi, where is my order A1001?")

print("tool calls:", len(calls))
for call in calls:
    print("  tool: ", call["name"])
    print("  input:", call["input"])  # the arguments, as a JSON string
if not calls:
    print("no tool call: the model answered in text instead (smaller models do this more often)")

tool calls: 1
  tool:  track_shipment
  input: {"order_id": "A1001"}


## Letting the agent run it

Give the same tools to an `Agent` and it runs the whole loop: call the model, run the tool it asks for,
hand back the result, call the model again, until it answers in text. The conversation it builds shows
each step: a `toolUse` block from the model, then a `toolResult` block carrying the tool's output.

In [8]:
# Give the same tools to an Agent. callback_handler=None keeps it from printing while it works.
agent = Agent(model=model, tools=tools, system_prompt=SPECIALIST, callback_handler=None)
result = agent("Hi, where is my order A1001?")  # runs the whole loop and returns the final result

`agent.messages` is the conversation the agent built. Print one row per block, so you can follow the loop
from the customer's message to the final answer.

In [9]:
print(f"{'role':9} {'block':10} content")
for message in agent.messages:
    for block in message["content"]:
        if "toolUse" in block:  # the model asking for a tool
            print(f"{message['role']:9} toolUse    {block['toolUse']['name']} {block['toolUse']['input']}")
        elif "toolResult" in block:  # what the tool returned
            print(f"{message['role']:9} toolResult {block['toolResult']['content'][0]['text'][:90]} ...")
        elif "text" in block:  # plain text: the customer's message or the model's answer, on one line
            print(f"{message['role']:9} text       {' '.join(block['text'].split())[:200]}")

role      block      content
user      text       Hi, where is my order A1001?
assistant toolUse    track_shipment {'order_id': 'A1001'}
user      toolResult {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "deli ...
assistant text       Your order A1001 is currently in transit with LBC. It is expected to arrive by November 4, 2026. The latest tracking events are: - October 28, 2026: Order packed at Manila warehouse - October 30, 2026


That is the whole idea of an agent: in LangGraph and CrewAI notebook 02 you ran this loop by hand
first; in Strands the agent is the starting point. Notebook 03 looks at the loop more closely.

## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario, read from the stream
as above. Nothing is run.

In [10]:
# Show the model's first move for the opening message of every ShipFast scenario. Nothing is run.
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
with_tool = 0  # how many messages started with a tool call
print(f"{'customer message':50} -> first move")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    calls = await first_move(text)
    asked = ", ".join(f"{c['name']}({c['input']})" for c in calls) or "(no tool)"
    with_tool += bool(calls)
    print(f"{text[:48]:50} -> {asked}")  # long messages are cut to 48 characters

print(f"\n{with_tool} of {len(seen)} messages started with a tool call")

customer message                                   -> first move


Hi, where is my order A1001?                       -> track_shipment({"order_id": "A1001"})


My order A1002 never arrived and tracking says i   -> lookup_order({"order_id": "A1002"})


Order A1003 is more than a week late. Please can   -> lookup_order({"order_id": "A1003"})


My vase set from order A1004 arrived broken.       -> lookup_order({"order_id": "A1004"})


I want to return order A1005, the stand wobbles.   -> lookup_order({"order_id": "A1005"})


Where is order A9999?                              -> lookup_order({"order_id": "A9999"})


Can you write me a poem about cats?                -> (no tool)

6 of 7 messages started with a tool call


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all.

The instructions' first step is "look up the order with lookup_order", and when we ran it, Strands' model
followed it on five of the six in-scope messages. With the same instructions and model, the LangGraph and
CrewAI runs mostly opened with `track_shipment` instead. Each framework packages the system prompt, the
tool specs and the messages a little differently, and a small model notices.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [11]:
# A fresh agent, asked about an order that does not exist.
shipfast.reset()
result = Agent(model=model, tools=tools, system_prompt=SPECIALIST, callback_handler=None)("Where is order A9999?")

print("tools called:", ", ".join(shipfast.tool_calls()) or "none")
print("reply:       ", str(result).strip())

tools called: lookup_order
reply:        The order ID A9999 could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In one of our early test runs, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, or above what is left after earlier refunds, whoever calls it.

In [12]:
# Three refunds on A1002, an order that cost $45: too much, the full amount, then the full amount again.
shipfast.reset()
too_much = shipfast.issue_refund("A1002", 120, "lost package")
correct = shipfast.issue_refund("A1002", 45, "lost package")
again = shipfast.issue_refund("A1002", 45, "lost package, again")  # nothing is left to refund

print("refund of $120:       ", too_much)
print("refund of $45:        ", correct)
print("refund of $45 again:  ", again)

refund of $120:        {"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
refund of $45:         {"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}
refund of $45 again:   {"error": "amount_exceeds_remaining", "order_id": "A1002", "amount": 45.0, "already_refunded": 45.0, "remaining": 0.0}


Only one of those should have reached the ledger, the record of refunds that were really issued.

In [13]:
# refunds() is the ledger: one entry per refund that was actually issued.
print("ledger:")
for r in shipfast.refunds():
    print(f"  {r['refund_id']}  order {r['order_id']}  ${r['amount']:.2f}  {r['status']}  ({r['reason']})")

ledger:
  RF-A1002  order A1002  $45.00  issued  (lost package)


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [14]:
# Every call to a shared tool is logged, whichever framework made it. Read the log, then reset it.
print("tools called, in order:", ", ".join(shipfast.tool_calls()))
shipfast.reset()  # leave a clean state behind

tools called, in order: issue_refund, issue_refund, issue_refund


## How the others do this

- **LangGraph** (LangChain) wraps the same functions with `tool()` and attaches them with
  `llm.bind_tools(tools)`; calling the model returns tool calls you can run by hand, and `create_agent`
  runs the loop.
- **CrewAI** wraps them with its own `tool` decorator; `LLM.call(..., tools=...)` returns raw tool calls,
  and an `Agent` runs the loop.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a spec the model reads; keep the logic outside the framework.
- The model only *asks* for a tool; in Strands the `Agent` runs it and hands back the result.
- `agent.messages` shows every `toolUse` and `toolResult`, so the loop is never hidden.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**.